In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=8e2be07a5042e32d0af61f8516936fb5e6af3d560fd4708260d81bf853cd25fd
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
# ==============================================================================
# STAGE 2 DICTABERT-LARGE: QA / REFERENT EXTRACTION
# ==============================================================================
import os
import re
import shutil
import time
import inspect
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    Trainer,
    TrainingArguments,
)
from google.colab import drive

# ------------------------------------------------------------------------------
# 1. CONFIGURATION & PATHS
# ------------------------------------------------------------------------------
drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert-large"
BASE_MODEL_PATH = "/content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert/"
LARGE_SAVE_PATH = "/content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert_large/"
TMP_CHECKPOINT_DIR = "/tmp/stage2_large_checkpoints"

TRAIN_PATH = "train_set.csv" if os.path.exists("train_set.csv") else "stage2_prompt_span_splits/train_set.csv"
VAL_PATH = "val_set.csv" if os.path.exists("val_set.csv") else "stage2_prompt_span_splits/val_set.csv"

os.makedirs(LARGE_SAVE_PATH, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
SPECIAL_TOKENS = {"additional_special_tokens": ["<mv>", "</mv>"]}
tokenizer.add_special_tokens(SPECIAL_TOKENS)

# ------------------------------------------------------------------------------
# 2. DATASET & HELPERS
# ------------------------------------------------------------------------------
class Stage2QADataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=160, is_train=True):
        self.df = pd.read_csv(csv_path).dropna(
            subset=["Full_Raw_Sentence", "MeaningVal", "Referent"]
        )
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_word = str(row.get("Prompt_Word", row["MeaningVal"])).strip()
        sentence = str(row["Full_Raw_Sentence"]).strip()
        meaning_val = str(row["MeaningVal"]).strip()
        referent = str(row["Referent"]).strip()

        marked_sentence = sentence.replace(meaning_val, f"<mv> {meaning_val} </mv>", 1) if meaning_val in sentence else sentence
        context = f"גירוי: {prompt_word} | משפט: {marked_sentence}"
        question = f"מהו הרפרנט של יחידת המשמעות: {meaning_val} ?"

        encoding = self.tokenizer(
            question,
            context,
            max_length=self.max_len,
            padding="max_length",
            truncation="only_second",
            return_offsets_mapping=True,
            return_token_type_ids=True,
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        token_type_ids = encoding["token_type_ids"].squeeze(0)

        ref_start_char = context.find(referent)
        ref_len = len(referent) if ref_start_char != -1 else len(prompt_word)
        if ref_start_char == -1:
            ref_start_char = context.find(prompt_word)

        if ref_start_char == -1:
            start_tok_idx, end_tok_idx = 0, 0
        else:
            ref_end_char = ref_start_char + ref_len - 1
            start_tok = encoding.char_to_token(0, ref_start_char, sequence_index=1)
            end_tok = encoding.char_to_token(0, ref_end_char, sequence_index=1)
            start_tok_idx = start_tok if start_tok is not None else 0
            end_tok_idx = end_tok if end_tok is not None else 0

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "start_positions": torch.tensor(start_tok_idx, dtype=torch.long),
            "end_positions": torch.tensor(end_tok_idx, dtype=torch.long),
            "raw_referent": referent,
            "raw_context": context,
        }

def normalize_hebrew_text(text):
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())

def extract_best_valid_span_fast(start_logits, end_logits, token_type_ids, max_span_len=12):
    seq_len = len(start_logits)
    context_mask = (token_type_ids == 1)
    score_matrix = start_logits[:, None] + end_logits[None, :]
    indices = np.arange(seq_len)
    i_idx, j_idx = np.meshgrid(indices, indices, indexing='ij')

    valid_mask = (
        (i_idx <= j_idx) &
        ((j_idx - i_idx + 1) <= max_span_len) &
        context_mask[:, None] &
        context_mask[None, :]
    )

    score_matrix[~valid_mask] = -1e9
    best_idx = np.unravel_index(np.argmax(score_matrix), score_matrix.shape)
    return best_idx[0], best_idx[1]

# ------------------------------------------------------------------------------
# 3. TRAINING STAGE 2 DICTABERT-LARGE
# ------------------------------------------------------------------------------
train_dataset = Stage2QADataset(TRAIN_PATH, tokenizer, is_train=True)
val_dataset = Stage2QADataset(VAL_PATH, tokenizer, is_train=False)

print(f"\n🚀 Loading model weights for {MODEL_NAME}...")
model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

batch_size = 8
num_epochs = 3
total_steps = (len(train_dataset) // (batch_size * 2)) * num_epochs
warmup_steps = int(0.10 * total_steps)

ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

training_args = TrainingArguments(**{
    eval_key: "epoch",
    "output_dir": TMP_CHECKPOINT_DIR,
    "save_strategy": "epoch",
    "learning_rate": 2.0e-5,
    "lr_scheduler_type": "cosine",
    "per_device_train_batch_size": batch_size,
    "per_device_eval_batch_size": batch_size,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": num_epochs,
    "warmup_steps": warmup_steps,
    "weight_decay": 0.01,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 50,
    "save_total_limit": 1,
    "save_only_model": True,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "report_to": "none",
})

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\n🚀 Training DictaBERT-Large...")
trainer.train()

print(f"\n💾 Saving final clean model and tokenizer to Google Drive...")
model.save_pretrained(LARGE_SAVE_PATH)
tokenizer.save_pretrained(LARGE_SAVE_PATH)

shutil.rmtree(TMP_CHECKPOINT_DIR, ignore_errors=True)
print(f"✅ MODEL SUCCESSFULLY SAVED TO DRIVE: {LARGE_SAVE_PATH}")

# ------------------------------------------------------------------------------
# 4. COMPARISON EVALUATION (BASE VS. LARGE)
# ------------------------------------------------------------------------------
def evaluate_stage2_path(model_path, model_label, batch_size=8):
    if not os.path.exists(model_path):
        print(f"⚠️ Directory missing: {model_path}")
        return None

    print(f"\n⏳ Evaluating {model_label}...")

    try:
        eval_tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    except Exception:
        hub_id = "dicta-il/dictabert-large" if "Large" in model_label else "dicta-il/dictabert"
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    eval_mod = AutoModelForQuestionAnswering.from_pretrained(
        model_path, trust_remote_code=True
    ).to(device)
    eval_mod.eval()

    eval_ds = Stage2QADataset(VAL_PATH, eval_tok, is_train=False)
    dataloader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)

    dummy_input = eval_tok(
        "מהו הרפרנט?", "גירוי: test | משפט: test", return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        _ = eval_mod(**dummy_input)

    token_f1_list, text_em_list, text_f1_list = [], [], []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            gold_starts = batch["start_positions"].numpy()
            gold_ends = batch["end_positions"].numpy()
            gold_raw_texts = batch["raw_referent"]

            outputs = eval_mod(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            batch_start_logits = outputs.start_logits.cpu().numpy()
            batch_end_logits = outputs.end_logits.cpu().numpy()
            batch_types = token_type_ids.cpu().numpy()

            for i in range(len(gold_starts)):
                p_start, p_end = extract_best_valid_span_fast(
                    batch_start_logits[i], batch_end_logits[i], batch_types[i]
                )
                g_start, g_end = gold_starts[i], gold_ends[i]

                p_tokens = input_ids[i][p_start : p_end + 1].cpu().tolist() if p_start <= p_end else []
                g_tokens = input_ids[i][g_start : g_end + 1].cpu().tolist() if g_start <= g_end else []

                p_counter, g_counter = Counter(p_tokens), Counter(g_tokens)
                num_same = sum((p_counter & g_counter).values())

                if len(p_tokens) == 0 or len(g_tokens) == 0:
                    tok_f1 = 1.0 if p_tokens == g_tokens else 0.0
                elif num_same == 0:
                    tok_f1 = 0.0
                else:
                    prec = num_same / len(p_tokens)
                    rec = num_same / len(g_tokens)
                    tok_f1 = 2 * (prec * rec) / (prec + rec)
                token_f1_list.append(tok_f1)

                pred_raw = eval_tok.decode(p_tokens, skip_special_tokens=True)
                pred_clean = normalize_hebrew_text(re.sub(r"</?mv>", "", pred_raw))
                gold_clean = normalize_hebrew_text(gold_raw_texts[i])

                txt_em = 1.0 if pred_clean == gold_clean else 0.0
                text_em_list.append(txt_em)

                p_words, g_words = pred_clean.split(), gold_clean.split()
                common_words = sum((Counter(p_words) & Counter(g_words)).values())
                if not p_words or not g_words:
                    txt_f1 = 1.0 if pred_clean == gold_clean else 0.0
                elif common_words == 0:
                    txt_f1 = 0.0
                else:
                    prec = common_words / len(p_words)
                    rec = common_words / len(g_words)
                    txt_f1 = 2 * (prec * rec) / (prec + rec)
                text_f1_list.append(txt_f1)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Model": model_label,
        "Token F1 ↑": f"{np.mean(token_f1_list)*100:.2f}%",
        "Text EM ↑": f"{np.mean(text_em_list)*100:.2f}%",
        "Text F1 ↑": f"{np.mean(text_f1_list)*100:.2f}%",
        "Latency ↓": f"{latency_ms:.2f} ms"
    }

base_results = evaluate_stage2_path(BASE_MODEL_PATH, "DictaBERT Base")
large_results = evaluate_stage2_path(LARGE_SAVE_PATH, "DictaBERT Large")

eval_data = [res for res in [base_results, large_results] if res is not None]
df_slide = pd.DataFrame(eval_data)

print("\n" + "=" * 65)
print("📊 STAGE 2 FINAL SLIDE COMPARISON TABLE")
print("=" * 65)
print(df_slide.to_markdown(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 Loading model weights for dicta-il/dictabert-large...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: dicta-il/dictabert-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance


🚀 Training DictaBERT-Large...


Epoch,Training Loss,Validation Loss
1,0.622454,0.651394
2,0.373495,0.715877
3,0.206707,0.910596


...


💾 Saving final clean model and tokenizer to Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ MODEL SUCCESSFULLY SAVED TO DRIVE: /content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert_large/
⚠️ Directory missing: /content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert/

⏳ Evaluating DictaBERT Large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


📊 STAGE 2 FINAL SLIDE COMPARISON TABLE
| Model           | Token F1 ↑   | Text EM ↑   | Text F1 ↑   | Latency ↓   |
|:----------------|:-------------|:------------|:------------|:------------|
| DictaBERT Large | 81.74%       | 70.67%      | 74.50%      | 34.89 ms    |


In [ ]:
BASE_MODEL_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/"
LARGE_SAVE_PATH = "/content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert_large/"

base_results = evaluate_stage2_path(BASE_MODEL_PATH, "DictaBERT Base")
large_results = evaluate_stage2_path(LARGE_SAVE_PATH, "DictaBERT Large")

eval_data = [res for res in [base_results, large_results] if res is not None]
df_slide = pd.DataFrame(eval_data)

print("\n" + "=" * 65)
print("📊 STAGE 2 FINAL SLIDE COMPARISON TABLE")
print("=" * 65)
print(df_slide.to_markdown(index=False))


⏳ Evaluating DictaBERT Base...


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]


⏳ Evaluating DictaBERT Large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


📊 STAGE 2 FINAL SLIDE COMPARISON TABLE
| Model           | Token F1 ↑   | Text EM ↑   | Text F1 ↑   | Latency ↓   |
|:----------------|:-------------|:------------|:------------|:------------|
| DictaBERT Base  | 80.97%       | 69.67%      | 73.44%      | 12.20 ms    |
| DictaBERT Large | 81.74%       | 70.67%      | 74.50%      | 34.59 ms    |


On Test

In [ ]:
# ==============================================================================
# EVALUATION ON TEST SET (BASE VS. LARGE)
# ==============================================================================
import os
import re
import time
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from google.colab import drive

# 1. SETUP & PATHS
drive.mount('/content/drive', force_remount=False)

BASE_MODEL_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/"
LARGE_SAVE_PATH = "/content/drive/MyDrive/Workshop/Stage_2/models/stage2_dictabert_large/"

TEST_PATH = "test_set.csv" if os.path.exists("test_set.csv") else "stage2_prompt_span_splits/test_set.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. DATASET & HELPERS
class Stage2QADataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=160, is_train=False):
        self.df = pd.read_csv(csv_path).dropna(
            subset=["Full_Raw_Sentence", "MeaningVal", "Referent"]
        )
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_word = str(row.get("Prompt_Word", row["MeaningVal"])).strip()
        sentence = str(row["Full_Raw_Sentence"]).strip()
        meaning_val = str(row["MeaningVal"]).strip()
        referent = str(row["Referent"]).strip()

        marked_sentence = sentence.replace(meaning_val, f"<mv> {meaning_val} </mv>", 1) if meaning_val in sentence else sentence
        context = f"גירוי: {prompt_word} | משפט: {marked_sentence}"
        question = f"מהו הרפרנט של יחידת המשמעות: {meaning_val} ?"

        encoding = self.tokenizer(
            question,
            context,
            max_length=self.max_len,
            padding="max_length",
            truncation="only_second",
            return_offsets_mapping=True,
            return_token_type_ids=True,
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        token_type_ids = encoding["token_type_ids"].squeeze(0)

        ref_start_char = context.find(referent)
        ref_len = len(referent) if ref_start_char != -1 else len(prompt_word)
        if ref_start_char == -1:
            ref_start_char = context.find(prompt_word)

        if ref_start_char == -1:
            start_tok_idx, end_tok_idx = 0, 0
        else:
            ref_end_char = ref_start_char + ref_len - 1
            start_tok = encoding.char_to_token(0, ref_start_char, sequence_index=1)
            end_tok = encoding.char_to_token(0, ref_end_char, sequence_index=1)
            start_tok_idx = start_tok if start_tok is not None else 0
            end_tok_idx = end_tok if end_tok is not None else 0

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "start_positions": torch.tensor(start_tok_idx, dtype=torch.long),
            "end_positions": torch.tensor(end_tok_idx, dtype=torch.long),
            "raw_referent": referent,
            "raw_context": context,
        }

def normalize_hebrew_text(text):
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())

def extract_best_valid_span_fast(start_logits, end_logits, token_type_ids, max_span_len=12):
    seq_len = len(start_logits)
    context_mask = (token_type_ids == 1)
    score_matrix = start_logits[:, None] + end_logits[None, :]
    indices = np.arange(seq_len)
    i_idx, j_idx = np.meshgrid(indices, indices, indexing='ij')

    valid_mask = (
        (i_idx <= j_idx) &
        ((j_idx - i_idx + 1) <= max_span_len) &
        context_mask[:, None] &
        context_mask[None, :]
    )

    score_matrix[~valid_mask] = -1e9
    best_idx = np.unravel_index(np.argmax(score_matrix), score_matrix.shape)
    return best_idx[0], best_idx[1]

# 3. EVALUATION FUNCTION
def evaluate_stage2_path(model_path, model_label, data_path, batch_size=8):
    if not os.path.exists(model_path):
        print(f"⚠️ Directory missing: {model_path}")
        return None

    print(f"\n⏳ Evaluating {model_label} on test set ({data_path})...")

    try:
        eval_tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    except Exception:
        hub_id = "dicta-il/dictabert-large" if "Large" in model_label else "dicta-il/dictabert"
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    eval_mod = AutoModelForQuestionAnswering.from_pretrained(
        model_path, trust_remote_code=True
    ).to(device)
    eval_mod.eval()

    eval_ds = Stage2QADataset(data_path, eval_tok, is_train=False)
    dataloader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)

    dummy_input = eval_tok(
        "מהו הרפרנט?", "גירוי: test | משפט: test", return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        _ = eval_mod(**dummy_input)

    token_f1_list, text_em_list, text_f1_list = [], [], []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            gold_starts = batch["start_positions"].numpy()
            gold_ends = batch["end_positions"].numpy()
            gold_raw_texts = batch["raw_referent"]

            outputs = eval_mod(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            batch_start_logits = outputs.start_logits.cpu().numpy()
            batch_end_logits = outputs.end_logits.cpu().numpy()
            batch_types = token_type_ids.cpu().numpy()

            for i in range(len(gold_starts)):
                p_start, p_end = extract_best_valid_span_fast(
                    batch_start_logits[i], batch_end_logits[i], batch_types[i]
                )
                g_start, g_end = gold_starts[i], gold_ends[i]

                p_tokens = input_ids[i][p_start : p_end + 1].cpu().tolist() if p_start <= p_end else []
                g_tokens = input_ids[i][g_start : g_end + 1].cpu().tolist() if g_start <= g_end else []

                p_counter, g_counter = Counter(p_tokens), Counter(g_tokens)
                num_same = sum((p_counter & g_counter).values())

                if len(p_tokens) == 0 or len(g_tokens) == 0:
                    tok_f1 = 1.0 if p_tokens == g_tokens else 0.0
                elif num_same == 0:
                    tok_f1 = 0.0
                else:
                    prec = num_same / len(p_tokens)
                    rec = num_same / len(g_tokens)
                    tok_f1 = 2 * (prec * rec) / (prec + rec)
                token_f1_list.append(tok_f1)

                pred_raw = eval_tok.decode(p_tokens, skip_special_tokens=True)
                pred_clean = normalize_hebrew_text(re.sub(r"</?mv>", "", pred_raw))
                gold_clean = normalize_hebrew_text(gold_raw_texts[i])

                txt_em = 1.0 if pred_clean == gold_clean else 0.0
                text_em_list.append(txt_em)

                p_words, g_words = pred_clean.split(), gold_clean.split()
                common_words = sum((Counter(p_words) & Counter(g_words)).values())
                if not p_words or not g_words:
                    txt_f1 = 1.0 if pred_clean == gold_clean else 0.0
                elif common_words == 0:
                    txt_f1 = 0.0
                else:
                    prec = common_words / len(p_words)
                    rec = common_words / len(g_words)
                    txt_f1 = 2 * (prec * rec) / (prec + rec)
                text_f1_list.append(txt_f1)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Model": model_label,
        "Token F1 ↑": f"{np.mean(token_f1_list)*100:.2f}%",
        "Text EM ↑": f"{np.mean(text_em_list)*100:.2f}%",
        "Text F1 ↑": f"{np.mean(text_f1_list)*100:.2f}%",
        "Latency ↓": f"{latency_ms:.2f} ms"
    }

# 4. RUN COMPARISON
base_results = evaluate_stage2_path(BASE_MODEL_PATH, "DictaBERT Base", TEST_PATH)
large_results = evaluate_stage2_path(LARGE_SAVE_PATH, "DictaBERT Large", TEST_PATH)

eval_data = [res for res in [base_results, large_results] if res is not None]
df_slide = pd.DataFrame(eval_data)

print("\n" + "=" * 65)
print("📊 STAGE 2 TEST SET COMPARISON TABLE")
print("=" * 65)
print(df_slide.to_markdown(index=False))

Mounted at /content/drive

⏳ Evaluating DictaBERT Base on test set (test_set.csv)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


⏳ Evaluating DictaBERT Large on test set (test_set.csv)...


Loading weights:   0%|          | 0/391 [00:02<?, ?it/s]


📊 STAGE 2 TEST SET COMPARISON TABLE
| Model           | Token F1 ↑   | Text EM ↑   | Text F1 ↑   | Latency ↓   |
|:----------------|:-------------|:------------|:------------|:------------|
| DictaBERT Base  | 79.64%       | 69.49%      | 73.12%      | 10.32 ms    |
| DictaBERT Large | 81.25%       | 71.15%      | 74.88%      | 29.81 ms    |
